
# GPU V4 — Candidate Preselection + Adaptive Search (Final Clean)

Notebook này là **bản rút gọn cuối cùng của GPU V4**, chỉ giữ các phần cần thiết để:

1. giải thích rõ cơ chế V4;
2. kiểm tra correctness;
3. chứng minh V4 khác dense NLM ở đâu;
4. đo mức giảm candidate;
5. so sánh chất lượng và runtime với GPU V3;
6. tạo các bảng/hình có thể dùng trực tiếp trong báo cáo hoặc slide.

V4 là một **adaptive / approximate branch**: nó chủ động thay đổi candidate set để giảm số full patch comparisons. Vì vậy output V4 final **không bắt buộc khớp tuyệt đối CPU/V3**.

Cấu hình benchmark cuối:

- Ảnh: `512×512`
- Patch: `7×7`
- Search window đầy đủ: `21×21`
- CUDA block: `16×16`
- `h = 0.12`
- `edge_search_window_size = 11`
- `texture_variance_threshold = 0.0025`
- `mean_threshold = 0.10`
- `variance_ratio_threshold = 4.0`



## 1. Ý tưởng V4

GPU V4 giữ công thức distance từ V3:

\[
\|P-Q\|^2
=
\|P\|^2+\|Q\|^2-2\langle P,Q\rangle
\]

nhưng **không tạo dense Product Map cho toàn bộ displacement** trước khi pruning.

Thay vào đó:

- precompute `Patch Energy`, `Patch Mean`, `Patch Variance`;
- mỗi CUDA thread sở hữu một output pixel;
- dùng variance của reference patch để chọn search window;
- dùng mean và variance để loại candidate rẻ;
- chỉ candidate còn lại mới thực hiện full patch comparison / dot product.

Điểm cốt lõi:

> **V3 làm mỗi comparison rẻ hơn; V4 cố gắng không thực hiện những comparison không cần thiết.**



### 1.1. Flow đầy đủ của V4

```text
Input noisy image
        │
        ▼
Reflect padding
        │
        ▼
┌───────────────────────────────────┐
│ Precompute patch statistics       │
│                                   │
│  Energy Map                       │
│  Mean Map                         │
│  Variance Map                     │
└───────────────────────────────────┘
        │
        ▼
CUDA Grid
  └── Block
       └── Thread
            └── 1 thread = 1 output pixel
                         │
                         ▼
              Read reference statistics
                         │
                         ▼
              ┌─────────────────────┐
              │ Adaptive Search     │
              │                     │
              │ ref variance >= T ? │
              └─────────────────────┘
                   │           │
                 Yes           No
                   │           │
                   ▼           ▼
               11×11         21×21
               search        search
                   └──────┬────┘
                          ▼
                 Candidate Q
                          │
                          ▼
              Mean difference test
              |μP - μQ| <= Tmean ?
                    │
             ┌──────┴──────┐
             │             │
           Fail           Pass
             │             │
             ▼             ▼
           Reject   Variance-ratio test
                    ratio <= Tvar ?
                          │
                   ┌──────┴──────┐
                   │             │
                 Fail           Pass
                   │             │
                   ▼             ▼
                 Reject      Full dot product
                              <P,Q>
                                 │
                                 ▼
                         Patch distance
                                 │
                                 ▼
                           NLM weight
                                 │
                                 ▼
                     Weighted accumulation
                                 │
                                 ▼
                         Normalize output
                                 │
                                 ▼
                          Output pixel
```

Không cần atomic update vì mỗi thread chỉ ghi output pixel của chính nó.


In [ ]:

# Sơ đồ trực quan bằng Matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(12, 14))
ax.set_xlim(0, 12)
ax.set_ylim(0, 18)
ax.axis("off")

def box(x, y, w, h, text, fontsize=10):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03",
        fill=False,
        linewidth=1.5,
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(x1, y1, x2, y2):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", linewidth=1.3),
    )

box(3.5, 16.5, 5, 0.8, "Input noisy image")
arrow(6, 16.5, 6, 15.8)
box(3.5, 15.0, 5, 0.8, "Reflect padding")
arrow(6, 15.0, 6, 14.3)
box(2.5, 13.2, 7, 1.1, "Precompute Patch Statistics\nEnergy Map • Mean Map • Variance Map")
arrow(6, 13.2, 6, 12.5)
box(3.0, 11.4, 6, 1.1, "CUDA mapping\n1 thread → 1 output pixel")
arrow(6, 11.4, 6, 10.7)
box(3.0, 9.6, 6, 1.1, "Adaptive Search\nreference variance vs texture threshold")
arrow(4.5, 9.6, 3.3, 8.8)
arrow(7.5, 9.6, 8.7, 8.8)
box(1.3, 7.9, 4, 0.9, "High variance / edge\n→ 11×11 search")
box(6.7, 7.9, 4, 0.9, "Low variance / smooth\n→ 21×21 search")
arrow(3.3, 7.9, 5.3, 7.1)
arrow(8.7, 7.9, 6.7, 7.1)
box(3.2, 6.2, 5.6, 0.9, "Candidate Q")
arrow(6, 6.2, 6, 5.5)
box(2.7, 4.5, 6.6, 1.0, "Mean test\n|μP - μQ| ≤ Tmean")
arrow(6, 4.5, 6, 3.8)
box(2.7, 2.8, 6.6, 1.0, "Variance-ratio test\nmax(varP,varQ) / min(varP,varQ) ≤ Tvar")
arrow(6, 2.8, 6, 2.1)
box(3.1, 1.0, 5.8, 1.1, "Accepted candidate\n→ direct <P,Q> → distance → weight")
arrow(6, 1.0, 6, 0.4)
ax.text(6, 0.15, "Weighted accumulation → normalize → output pixel", ha="center", fontsize=10)

plt.title("GPU V4 — Luồng xử lý đầy đủ", fontsize=14)
plt.show()



## 2. Ý nghĩa ba ngưỡng của V4

### 2.1. `texture_variance_threshold`

Dùng **variance của reference patch** để quyết định kích thước search window:

\[
\sigma_P^2 \ge T_{\text{texture}}
\]

- đúng → vùng edge/texture → search nhỏ `11×11`;
- sai → vùng smooth → giữ full search `21×21`.

Ngưỡng này **không so sánh hai patch với nhau**.

### 2.2. `mean_threshold`

Với reference patch \(P\) và candidate \(Q\):

\[
|\mu_P-\mu_Q| \le T_{\mu}
\]

Nếu không đạt, candidate bị loại ngay.

### 2.3. `variance_ratio_threshold`

\[
\frac{\max(\sigma_P^2,\sigma_Q^2)+\epsilon}
{\min(\sigma_P^2,\sigma_Q^2)+\epsilon}
\le T_{\sigma}
\]

Nếu không đạt, candidate bị loại trước full patch comparison.



## 3. Thiết lập môi trường và dữ liệu


In [ ]:

from pathlib import Path
import sys
import numpy as np
import cupy as cp
import pandas as pd
import time

from skimage import io, color, transform
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nlm.cpu import nlm_cpu_naive
from nlm.gpu_v3 import (
    nlm_gpu_v3,
    prepare_gpu_v3_pipeline_launch,
)
from nlm.gpu_v4 import (
    nlm_gpu_v4,
    prepare_gpu_v4_pipeline_launch,
)
from nlm.benchmark import (
    benchmark_cuda_kernel,
    benchmark_gpu_end_to_end,
)

print("ROOT:", ROOT)
print("CuPy:", cp.__version__)

props = cp.cuda.runtime.getDeviceProperties(0)
name = props["name"]
if isinstance(name, bytes):
    name = name.decode()
print("CUDA device:", name)


In [ ]:

IMAGE_NAME = "02.png"

image_size = (512, 512)
patch_size = 7
search_window_size = 21
block_size = (16, 16)
h = 0.12

noise_sigma = 0.08
rng_seed = 42

v4_params = dict(
    edge_search_window_size=11,
    texture_variance_threshold=0.0025,
    mean_threshold=0.10,
    variance_ratio_threshold=4.0,
)

print("V4 final configuration:")
print(v4_params)


In [ ]:

image_path = ROOT / "data" / "input" / IMAGE_NAME

clean = io.imread(image_path)

if clean.ndim == 3:
    clean = color.rgb2gray(clean)

clean = transform.resize(
    clean,
    image_size,
    anti_aliasing=True,
).astype(np.float32)

if clean.max() > 1.0:
    clean /= 255.0

rng = np.random.default_rng(rng_seed)

noisy = np.clip(
    clean
    + rng.normal(
        0.0,
        noise_sigma,
        clean.shape,
    ).astype(np.float32),
    0.0,
    1.0,
).astype(np.float32)

print("Clean shape:", clean.shape)
print("Clean range:", float(clean.min()), float(clean.max()))
print("Noisy range:", float(noisy.min()), float(noisy.max()))


In [ ]:

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(clean, cmap="gray", vmin=0, vmax=1)
plt.title("Ảnh sạch")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(noisy, cmap="gray", vmin=0, vmax=1)
plt.title("Ảnh nhiễu")
plt.axis("off")

plt.tight_layout()
plt.show()



## 4. Chạy V4 final và đọc thống kê candidate

`nlm_gpu_v4(..., return_stats=True)` trả về các metric sau:

| Metric | Ý nghĩa |
|---|---|
| `considered_candidates` | Tổng số candidate thực sự nằm trong các local search windows sau **Adaptive Search**, cộng trên toàn ảnh. |
| `accepted_candidates` | Tổng số candidate vượt qua cả Mean test và Variance-ratio test, do đó phải thực hiện full patch comparison. |
| `acceptance_ratio` | `accepted_candidates / considered_candidates`. Tỷ lệ candidate sau Adaptive Search tiếp tục đi tới expensive full comparison. |
| `rejection_ratio` | `1 - acceptance_ratio`. Tỷ lệ candidate bị Candidate Preselection loại sau khi đã được Adaptive Search xem xét. |
| `mean_considered_per_pixel` | Số candidate trung bình mà mỗi output pixel phải duyệt sau Adaptive Search. Dense `21×21` sẽ là `441`. |
| `mean_accepted_per_pixel` | Số candidate trung bình mỗi output pixel thực sự phải full compare sau cả Adaptive Search và Candidate Preselection. |

Quan trọng:

- `rejection_ratio` **chỉ đo pruning của Candidate Preselection trên tập candidate đã được Adaptive Search giữ lại**.
- Muốn biết tổng mức giảm full comparison so với dense NLM, phải so `mean_accepted_per_pixel` với `441`.


In [ ]:

out_v4, stats_v4 = nlm_gpu_v4(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
    return_stats=True,
    **v4_params,
)

stats_table = pd.DataFrame([
    {
        "Metric": "considered_candidates",
        "Value": stats_v4["considered_candidates"],
        "Giải thích": "Tổng candidate còn lại sau Adaptive Search trên toàn ảnh",
    },
    {
        "Metric": "accepted_candidates",
        "Value": stats_v4["accepted_candidates"],
        "Giải thích": "Tổng candidate qua Mean + Variance filter và phải full compare",
    },
    {
        "Metric": "acceptance_ratio",
        "Value": stats_v4["acceptance_ratio"],
        "Giải thích": "Tỷ lệ accepted / considered",
    },
    {
        "Metric": "rejection_ratio",
        "Value": stats_v4["rejection_ratio"],
        "Giải thích": "Tỷ lệ bị Candidate Preselection loại trong considered set",
    },
    {
        "Metric": "mean_considered_per_pixel",
        "Value": stats_v4["mean_considered_per_pixel"],
        "Giải thích": "Candidate trung bình / pixel sau Adaptive Search",
    },
    {
        "Metric": "mean_accepted_per_pixel",
        "Value": stats_v4["mean_accepted_per_pixel"],
        "Giải thích": "Full comparisons trung bình / pixel sau cả hai bước pruning",
    },
])

display(stats_table)



## 5. Phân rã mức giảm candidate

Dense NLM với search `21×21` luôn có:

\[
441 \text{ candidates / output pixel}
\]

Ta tách mức giảm thành hai tầng:

### Tầng 1 — Adaptive Search

\[
441
\rightarrow
\text{mean\_considered\_per\_pixel}
\]

### Tầng 2 — Candidate Preselection

\[
\text{mean\_considered\_per\_pixel}
\rightarrow
\text{mean\_accepted\_per\_pixel}
\]

Tổng full-comparison reduction:

\[
1 -
\frac{\text{mean accepted per pixel}}{441}
\]


In [ ]:

dense_candidates = search_window_size ** 2

mean_considered = stats_v4["mean_considered_per_pixel"]
mean_accepted = stats_v4["mean_accepted_per_pixel"]

adaptive_reduction = (
    1.0 - mean_considered / dense_candidates
) * 100.0

preselection_reduction_within_considered = (
    1.0 - mean_accepted / mean_considered
) * 100.0

total_full_comparison_reduction = (
    1.0 - mean_accepted / dense_candidates
) * 100.0

candidate_reduction_df = pd.DataFrame([
    {
        "Stage": "Dense NLM",
        "Mean candidates / pixel": dense_candidates,
        "Reduction vs previous stage (%)": 0.0,
    },
    {
        "Stage": "Sau Adaptive Search",
        "Mean candidates / pixel": mean_considered,
        "Reduction vs previous stage (%)": adaptive_reduction,
    },
    {
        "Stage": "Sau Candidate Preselection",
        "Mean candidates / pixel": mean_accepted,
        "Reduction vs previous stage (%)": preselection_reduction_within_considered,
    },
])

display(candidate_reduction_df)

print(
    "Tổng mức giảm full patch comparisons so với dense NLM:",
    f"{total_full_comparison_reduction:.2f}%"
)


In [ ]:

plt.figure(figsize=(8, 5))

labels = [
    "Dense\n21×21",
    "Sau Adaptive\nSearch",
    "Sau Candidate\nPreselection",
]

values = [
    dense_candidates,
    mean_considered,
    mean_accepted,
]

plt.bar(labels, values)
plt.ylabel("Mean candidates / output pixel")
plt.title("GPU V4 — Candidate reduction qua từng giai đoạn")

for i, value in enumerate(values):
    plt.text(i, value, f"{value:.1f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()



## 6. Sanity check — tắt toàn bộ pruning

Mục tiêu của cell này là chứng minh kernel V4 **không tính distance sai**.

Ta vô hiệu hóa:

1. **Adaptive Search**
   - `edge_search_window_size = 21`
   - `texture_variance_threshold = 1e9`

2. **Mean filter**
   - `mean_threshold = 1e9`

3. **Variance-ratio filter**
   - `variance_ratio_threshold = 1e9`

Khi đó V4 phải xét đủ:

\[
21\times21=441
\]

candidate / pixel và output phải gần dense NLM / V3.

Nếu dense mode khớp V3 nhưng V4 final không khớp, sai khác của V4 final đến từ **intentional pruning**, không phải lỗi distance kernel.


In [ ]:

out_v4_dense, stats_v4_dense = nlm_gpu_v4(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
    edge_search_window_size=search_window_size,
    texture_variance_threshold=1e9,
    mean_threshold=1e9,
    variance_ratio_threshold=1e9,
    return_stats=True,
)

out_v3 = nlm_gpu_v3(
    noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
)

dense_diff = np.abs(out_v3 - out_v4_dense)

print("Dense-mode mean considered / pixel:",
      stats_v4_dense["mean_considered_per_pixel"])
print("Dense-mode mean accepted / pixel:",
      stats_v4_dense["mean_accepted_per_pixel"])
print("V3 vs V4 dense MAE:", float(dense_diff.mean()))
print("V3 vs V4 dense Max error:", float(dense_diff.max()))
print(
    "V3 vs V4 dense allclose:",
    np.allclose(out_v3, out_v4_dense, rtol=1e-4, atol=1e-4),
)



## 7. Correctness với CPU baseline

Nếu đã có CPU baseline full `512×512` lưu từ notebook tổng hợp, notebook này sẽ tận dụng trực tiếp:

```text
outputs/cpu_baseline_512_patch7_search21.npy
```

Nếu chưa có file đó, notebook fallback sang crop nhỏ `16×16` để tránh chạy CPU full quá lâu.

Kỳ vọng:

- CPU vs V4 dense mode → gần như khớp;
- CPU vs V4 final → không cần exact equality vì V4 final thay đổi candidate set.


In [ ]:

CPU_BASELINE_PATH = (
    ROOT
    / "outputs"
    / "cpu_baseline_512_patch7_search21.npy"
)

if CPU_BASELINE_PATH.exists():
    print("Đã tìm thấy CPU baseline full:", CPU_BASELINE_PATH)

    cpu_reference = np.load(CPU_BASELINE_PATH)

    dense_cpu_diff = np.abs(cpu_reference - out_v4_dense)
    final_cpu_diff = np.abs(cpu_reference - out_v4)

    cpu_correctness_df = pd.DataFrame([
        {
            "Comparison": "CPU vs V4 dense",
            "MAE": float(dense_cpu_diff.mean()),
            "Max error": float(dense_cpu_diff.max()),
            "allclose": bool(np.allclose(
                cpu_reference,
                out_v4_dense,
                rtol=1e-4,
                atol=1e-4,
            )),
        },
        {
            "Comparison": "CPU vs V4 final",
            "MAE": float(final_cpu_diff.mean()),
            "Max error": float(final_cpu_diff.max()),
            "allclose": bool(np.allclose(
                cpu_reference,
                out_v4,
                rtol=1e-4,
                atol=1e-4,
            )),
        },
    ])

else:
    print("Không tìm thấy CPU baseline full → dùng crop 16×16.")

    correctness_input = noisy[:16, :16].copy()

    cpu_reference = nlm_cpu_naive(
        correctness_input,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
    )

    v4_dense_small = nlm_gpu_v4(
        correctness_input,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
        edge_search_window_size=search_window_size,
        texture_variance_threshold=1e9,
        mean_threshold=1e9,
        variance_ratio_threshold=1e9,
    )

    v4_final_small = nlm_gpu_v4(
        correctness_input,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
        **v4_params,
    )

    dense_cpu_diff = np.abs(cpu_reference - v4_dense_small)
    final_cpu_diff = np.abs(cpu_reference - v4_final_small)

    cpu_correctness_df = pd.DataFrame([
        {
            "Comparison": "CPU vs V4 dense (16×16)",
            "MAE": float(dense_cpu_diff.mean()),
            "Max error": float(dense_cpu_diff.max()),
            "allclose": bool(np.allclose(
                cpu_reference,
                v4_dense_small,
                rtol=1e-4,
                atol=1e-4,
            )),
        },
        {
            "Comparison": "CPU vs V4 final (16×16)",
            "MAE": float(final_cpu_diff.mean()),
            "Max error": float(final_cpu_diff.max()),
            "allclose": bool(np.allclose(
                cpu_reference,
                v4_final_small,
                rtol=1e-4,
                atol=1e-4,
            )),
        },
    ])

display(cpu_correctness_df)



## 8. V3 vs V4 final — sai khác output

V3 là dense exact branch. V4 final là adaptive branch.

Do đó:

\[
V3 \neq V4_{\text{final}}
\]

là kết quả **được kỳ vọng**.

Điều cần đánh giá là:

- sai khác output có hợp lý không;
- PSNR / SSIM;
- runtime;
- candidate reduction.


In [ ]:

v3_v4_diff = np.abs(out_v3 - out_v4)

print("V3 vs V4 final MAE:", float(v3_v4_diff.mean()))
print("V3 vs V4 final Max error:", float(v3_v4_diff.max()))
print(
    "V3 vs V4 final allclose:",
    np.allclose(out_v3, out_v4, rtol=1e-4, atol=1e-4),
)



## 9. Chất lượng ảnh — PSNR / SSIM

Không nên đánh giá V4 chỉ bằng equality với V3.

V4 final cần được đánh giá theo trade-off:

\[
\text{runtime}
\quad \leftrightarrow \quad
\text{PSNR / SSIM}
\]


In [ ]:

def quality_metrics(reference, test):
    return {
        "PSNR (dB)": peak_signal_noise_ratio(
            reference,
            test,
            data_range=1.0,
        ),
        "SSIM": structural_similarity(
            reference,
            test,
            data_range=1.0,
        ),
    }

quality_df = pd.DataFrame([
    {"Version": "Noisy", **quality_metrics(clean, noisy)},
    {"Version": "GPU V3", **quality_metrics(clean, out_v3)},
    {"Version": "GPU V4 final", **quality_metrics(clean, out_v4)},
])

display(quality_df)


In [ ]:

plt.figure(figsize=(12, 4))

for i, (title, image) in enumerate([
    ("Noisy", noisy),
    ("GPU V3", out_v3),
    ("GPU V4 final", out_v4),
], start=1):
    plt.subplot(1, 3, i)
    plt.imshow(image, cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()



## 10. Benchmark GPU compute-only

Dùng CUDA Events và launcher đã chuẩn bị sẵn.

Tên **compute-only** được dùng thay cho “kernel-only” vì V3/V4 có thể gồm nhiều CUDA kernels trong pipeline.


In [ ]:

v3_prepared = prepare_gpu_v3_pipeline_launch(
    image=noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
    displacement_batch_size=8,
)

v4_prepared = prepare_gpu_v4_pipeline_launch(
    image=noisy,
    patch_size=patch_size,
    search_window_size=search_window_size,
    h=h,
    block_size=block_size,
    **v4_params,
)

v3_launcher = v3_prepared["kernel_launcher"]
v4_launcher = v4_prepared["kernel_launcher"]

for _ in range(15):
    v3_launcher()
    v4_launcher()

cp.cuda.Stream.null.synchronize()

v3_compute = benchmark_cuda_kernel(
    v3_launcher,
    warmup_runs=10,
    measured_runs=50,
)

v4_compute = benchmark_cuda_kernel(
    v4_launcher,
    warmup_runs=10,
    measured_runs=50,
)

print("V3 compute-only:", f"{v3_compute['mean_ms']:.4f} ms")
print("V4 compute-only:", f"{v4_compute['mean_ms']:.4f} ms")
print(
    "V4 speedup vs V3:",
    f"{v3_compute['mean_ms'] / v4_compute['mean_ms']:.3f}x"
)



## 11. Benchmark End-to-End

End-to-End bao gồm:

- CPU-side preparation;
- padding;
- H2D;
- GPU pipeline;
- D2H.

Đây là metric gần với thời gian sử dụng thực tế hơn compute-only.


In [ ]:

for _ in range(15):
    _ = nlm_gpu_v3(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    )

    _ = nlm_gpu_v4(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
        **v4_params,
    )

cp.cuda.Stream.null.synchronize()

v3_e2e = benchmark_gpu_end_to_end(
    function=lambda: nlm_gpu_v3(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
    ),
    warmup_runs=10,
    measured_runs=50,
)

v4_e2e = benchmark_gpu_end_to_end(
    function=lambda: nlm_gpu_v4(
        noisy,
        patch_size=patch_size,
        search_window_size=search_window_size,
        h=h,
        block_size=block_size,
        **v4_params,
    ),
    warmup_runs=10,
    measured_runs=50,
)

v3_e2e_ms = v3_e2e["mean_seconds"] * 1000.0
v4_e2e_ms = v4_e2e["mean_seconds"] * 1000.0

print("V3 E2E:", f"{v3_e2e_ms:.4f} ms")
print("V4 E2E:", f"{v4_e2e_ms:.4f} ms")
print("V4 E2E speedup vs V3:", f"{v3_e2e_ms / v4_e2e_ms:.3f}x")



## 12. Bảng tổng hợp V3 vs V4

Bảng này là bảng chính để dùng trong báo cáo/slide.

Ngoài runtime và quality, V4 phải báo cáo số candidate để giải thích **vì sao** nó nhanh hơn.


In [ ]:

v3_quality = quality_metrics(clean, out_v3)
v4_quality = quality_metrics(clean, out_v4)

final_summary = pd.DataFrame([
    {
        "Version": "GPU V3",
        "Compute-only (ms)": v3_compute["mean_ms"],
        "End-to-End (ms)": v3_e2e_ms,
        "PSNR (dB)": v3_quality["PSNR (dB)"],
        "SSIM": v3_quality["SSIM"],
        "Mean considered / pixel": dense_candidates,
        "Mean accepted / pixel": dense_candidates,
        "Full comparison reduction vs dense (%)": 0.0,
    },
    {
        "Version": "GPU V4 final",
        "Compute-only (ms)": v4_compute["mean_ms"],
        "End-to-End (ms)": v4_e2e_ms,
        "PSNR (dB)": v4_quality["PSNR (dB)"],
        "SSIM": v4_quality["SSIM"],
        "Mean considered / pixel": mean_considered,
        "Mean accepted / pixel": mean_accepted,
        "Full comparison reduction vs dense (%)":
            total_full_comparison_reduction,
    },
])

display(final_summary)

print(
    "Compute speedup V4 vs V3:",
    f"{v3_compute['mean_ms'] / v4_compute['mean_ms']:.3f}x"
)

print(
    "E2E speedup V4 vs V3:",
    f"{v3_e2e_ms / v4_e2e_ms:.3f}x"
)



## 13. Kết luận

GPU V4 tối ưu NLM bằng hai tầng giảm candidate:

1. **Adaptive Search** giảm phạm vi tìm kiếm dựa trên variance của reference patch.
2. **Candidate Preselection** dùng mean và variance để loại candidate trước expensive full comparison.

Flow rút gọn:

```text
441 dense candidates / pixel
        ↓ Adaptive Search
mean_considered_per_pixel
        ↓ Mean + Variance filters
mean_accepted_per_pixel
        ↓
chỉ phần còn lại mới full compare
```

Sanity check dense mode phải gần V3/CPU, chứng minh kernel distance vẫn đúng.

V4 final có thể khác V3/CPU vì candidate set đã thay đổi có chủ đích. Do đó V4 được đánh giá bằng **runtime + candidate reduction + PSNR/SSIM**, thay vì chỉ dựa vào exact equality.

### Câu chốt khi trình bày

> V3 giảm chi phí của mỗi patch comparison, còn V4 giảm số patch comparisons phải thực hiện. V4 dùng local variance để chọn phạm vi search và dùng patch mean/variance để reject candidate rẻ trước khi tính full distance.
